In [2]:
!pip install -q -U torchao pylcs faiss-cpu rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 87.2 MB/s eta 0:00:00


In [3]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

Mounted at /content/drive
STATE RESTORED: 36501 corpus | 2618 LLM answers | all caches loaded


In [4]:
# ================================================================
# FINAL PROPOSALS — Run after bootstrap cell (all cached data loaded)
# Zero GPU. ~30 minutes CPU total. Split-half gated.
# ================================================================
import re, json, time, numpy as np, pandas as pd
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
# ---- CONSTANTS (should already exist from bootstrap) ----
TEST_MIX = {'Eng_Uga':0.284,'Aka_Gha':0.188,'Eng_Gha':0.188,'Lug_Uga':0.143,
            'Swa_Ken':0.087,'Eng_Ken':0.064,'Amh_Eth':0.023,'Eng_Eth':0.023}
ENGLISH_SUBS = ['Eng_Gha', 'Eng_Uga', 'Eng_Ken', 'Eng_Eth']
# ---- UNICODE ROUGE (should already exist from bootstrap) ----
UNI_RE = re.compile(r'\w+', re.UNICODE)
def _uni_toks(text):
    return UNI_RE.findall(str(text).lower())
def _uni_r1(ref_toks, hyp_toks):
    if not ref_toks or not hyp_toks: return 0.0
    rc = defaultdict(int)
    hc = defaultdict(int)
    for t in ref_toks: rc[t] += 1
    for t in hyp_toks: hc[t] += 1
    overlap = sum(min(rc[t], hc[t]) for t in rc)
    p = overlap / len(hyp_toks)
    r = overlap / len(ref_toks)
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0
def _uni_rl(ref_toks, hyp_toks):
    if not ref_toks or not hyp_toks: return 0.0
    m, n = len(ref_toks), len(hyp_toks)
    if m > 2000 or n > 2000:
        # fallback for very long texts
        return _uni_r1(ref_toks, hyp_toks) * 0.9
    prev = [0]*(n+1)
    for i in range(1, m+1):
        cur = [0]*(n+1)
        for j in range(1, n+1):
            if ref_toks[i-1] == hyp_toks[j-1]:
                cur[j] = prev[j-1]+1
            else:
                cur[j] = max(prev[j], cur[j-1])
        prev = cur
    lcs = prev[n]
    p = lcs/n
    r = lcs/m
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0
# Use existing functions if available, else use these
try:
    uni_toks; uni_r1; uni_rl
    print("Using existing ROUGE functions")
except NameError:
    uni_toks = _uni_toks; uni_r1 = _uni_r1; uni_rl = _uni_rl
    print("Defined ROUGE functions")
# ================================================================
# HELPERS
# ================================================================
def score_weighted(per_lang_scores, metric='r1'):
    """Compute test-mix weighted average from per-language dict of lists."""
    total = 0.0
    for sub, w in TEST_MIX.items():
        vals = per_lang_scores.get(sub, [])
        total += w * (np.mean(vals) if vals else 0.0)
    return total
def split_half_indices(val_df, seed=42):
    """Split val indices into two halves per language (stratified)."""
    rng = np.random.default_rng(seed)
    half_a, half_b = [], []
    for sub in SUBSET_TO_LANG:
        idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub]
        rng.shuffle(idxs)
        mid = len(idxs)//2
        half_a.extend(idxs[:mid])
        half_b.extend(idxs[mid:])
    return half_a, half_b
log = lambda msg: print(f"[{time.strftime('%H:%M:%S')}] {msg}")
# ================================================================
# PROPOSAL 1: CROSS-SUBSET ENGLISH RETRIEVAL
# ================================================================
log("="*60)
log("PROPOSAL 1: Cross-Subset English Retrieval for Eng_Gha")
log("="*60)
# Build combined English index from all English subsets
combined_subs = combined['subset'].values.astype(str)
combined_answers = combined['output'].fillna('').values.astype(str)
combined_questions = combined['input'].fillna('').values.astype(str)
# Get indices for all English training/val data
eng_global_idx = np.array([i for i in range(len(combined)) if combined_subs[i] in ENGLISH_SUBS])
log(f"English pool: {len(eng_global_idx)} answers ({', '.join(f'{s}:{(combined_subs[eng_global_idx]==s).sum()}' for s in ENGLISH_SUBS)})")
# Current same-subset pool for Eng_Gha
gha_only_idx = np.array([i for i in range(len(combined)) if combined_subs[i] == 'Eng_Gha'])
log(f"Eng_Gha only: {len(gha_only_idx)} answers")
log(f"Expansion: {len(eng_global_idx)/len(gha_only_idx):.1f}x more candidates")
# Get embeddings for English pool (use base AfriE5 embeddings)
# These should be the corpus embeddings from the bootstrap
try:
    # Try to use existing embeddings
    eng_embs = corpus_emb[eng_global_idx]
    log(f"English embeddings: {eng_embs.shape}")
except Exception as e:
    log(f"Need to load embeddings: {e}")
    log("Attempting to load from cached files...")
    # The user's bootstrap should have loaded these
    raise RuntimeError("corpus_emb not available. Run bootstrap first.")
# Build FAISS index for all English
import faiss
eng_embs_f32 = eng_embs.astype(np.float32).copy()
faiss.normalize_L2(eng_embs_f32)
eng_index = faiss.IndexFlatIP(eng_embs_f32.shape[1])
eng_index.add(eng_embs_f32)
log(f"English FAISS index: {eng_index.ntotal} vectors")
def cross_english_retrieve(query_emb, query_text, top_k=10):
    """Retrieve from ALL English subsets."""
    qe = query_emb.astype(np.float32).reshape(1, -1).copy()
    faiss.normalize_L2(qe)
    D, I = eng_index.search(qe, min(top_k * 3, eng_index.ntotal))
    results = []
    for j in range(I.shape[1]):
        real_idx = eng_global_idx[int(I[0][j])]
        q = str(combined_questions[real_idx]).strip()
        if q.lower() == query_text.strip().lower():
            continue  # skip exact question match
        results.append({
            'answer': str(combined_answers[real_idx]),
            'question': q,
            'subset': combined_subs[real_idx],
            'score': float(D[0][j]),
        })
        if len(results) >= top_k:
            break
    return results
# Evaluate on val (Eng_Gha only)
val_gha_idx = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])=='Eng_Gha']
log(f"Eng_Gha val queries: {len(val_gha_idx)}")
baseline_r1, baseline_rl = [], []
cross_r1, cross_rl = [], []
cross_source_stats = defaultdict(int)
for i in tqdm(val_gha_idx, desc="P1: Cross-English"):
    q = str(val_df.iloc[i]['input']).strip()
    ref = str(val_df.iloc[i]['output']).strip()
    rt = uni_toks(ref)
    if not rt: continue

    # Baseline: same-subset retrieval
    try:
        cands = val_cands_all[i]
        if cands:
            bl = uni_toks(cands[0]['answer'])
            baseline_r1.append(uni_r1(rt, bl))
            baseline_rl.append(uni_rl(rt, bl))
        else:
            baseline_r1.append(0.0)
            baseline_rl.append(0.0)
    except:
        baseline_r1.append(0.0)
        baseline_rl.append(0.0)

    # Cross-English retrieval
    cross_cands = cross_english_retrieve(val_emb[i], q, top_k=5)
    if cross_cands:
        # Score each candidate and pick best
        best_r1, best_rl, best_src = 0, 0, ''
        for c in cross_cands:
            ct = uni_toks(c['answer'])
            r1 = uni_r1(rt, ct)
            rl = uni_rl(rt, ct)
            if r1 > best_r1:
                best_r1, best_rl, best_src = r1, rl, c['subset']
        cross_r1.append(best_r1)
        cross_rl.append(best_rl)
        cross_source_stats[best_src] += 1
    else:
        cross_r1.append(0.0)
        cross_rl.append(0.0)
bl_r1_mean = np.mean(baseline_r1)
bl_rl_mean = np.mean(baseline_rl)
cr_r1_mean = np.mean(cross_r1)
cr_rl_mean = np.mean(cross_rl)
log(f"\nEng_Gha Results (top-1 for baseline, oracle-top-5 for cross):")
log(f"  Baseline (same-subset): R1={bl_r1_mean:.4f}  RL={bl_rl_mean:.4f}")
log(f"  Cross-English (best-5): R1={cr_r1_mean:.4f}  RL={cr_rl_mean:.4f}")
log(f"  Delta:                  R1={cr_r1_mean-bl_r1_mean:+.4f}  RL={cr_rl_mean-bl_rl_mean:+.4f}")
log(f"\n  Best answer sources: {dict(cross_source_stats)}")
log(f"  Answers from OTHER subsets: {sum(v for k,v in cross_source_stats.items() if k!='Eng_Gha')}/{len(val_gha_idx)}")
# Test-weighted impact (with 25-50% Gha transfer discount)
raw_delta_r1 = cr_r1_mean - bl_r1_mean
disc_delta_r1 = raw_delta_r1 * 0.375  # midpoint of 25-50%
weighted_r1 = disc_delta_r1 * 0.188 * 0.37
log(f"\n  Raw R1 delta on Eng_Gha: {raw_delta_r1:+.4f}")
log(f"  Discounted (37.5%):     {disc_delta_r1:+.4f}")
log(f"  Test-weighted impact:   {weighted_r1:+.5f}")
log(f"  GATE: {'PASS ✅' if weighted_r1 >= 0.002 else 'FAIL ❌'} (threshold: 0.002)")
# Also check: what if we use cross-English top-1 (not oracle)?
cross_top1_r1 = []
for i, idx in enumerate(val_gha_idx):
    q = str(val_df.iloc[idx]['input']).strip()
    ref = str(val_df.iloc[idx]['output']).strip()
    rt = uni_toks(ref)
    if not rt:
        cross_top1_r1.append(0.0)
        continue
    cross_cands = cross_english_retrieve(val_emb[idx], q, top_k=1)
    if cross_cands:
        ct = uni_toks(cross_cands[0]['answer'])
        cross_top1_r1.append(uni_r1(rt, ct))
    else:
        cross_top1_r1.append(0.0)
log(f"\n  Cross-English TOP-1 R1: {np.mean(cross_top1_r1):.4f} (vs baseline {bl_r1_mean:.4f}, delta {np.mean(cross_top1_r1)-bl_r1_mean:+.4f})")
print("\n")
# ================================================================
# PROPOSAL 2: ANSWER LENGTH CALIBRATION
# ================================================================
log("="*60)
log("PROPOSAL 2: Per-Language Answer Length Calibration")
log("="*60)
# For each language, compute optimal word count truncation
for sub in sorted(SUBSET_TO_LANG.keys()):
    sub_idx = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub]
    if not sub_idx:
        continue

    # Get reference lengths
    ref_lens = []
    for i in sub_idx:
        ref = str(val_df.iloc[i]['output']).strip()
        ref_lens.append(len(uni_toks(ref)))

    # Get current answer lengths (from candidates)
    ans_lens = []
    for i in sub_idx:
        try:
            if val_cands_all[i]:
                ans_lens.append(len(uni_toks(val_cands_all[i][0]['answer'])))
            else:
                ans_lens.append(0)
        except:
            ans_lens.append(0)

    # Compute ROUGE at different truncation levels
    best_trunc = -1  # -1 = no truncation
    best_f1 = 0
    no_trunc_f1s = []

    for i in sub_idx:
        ref = str(val_df.iloc[i]['output']).strip()
        rt = uni_toks(ref)
        try:
            ans = val_cands_all[i][0]['answer'] if val_cands_all[i] else ''
        except:
            ans = ''
        at = uni_toks(ans)
        if rt and at:
            no_trunc_f1s.append(uni_r1(rt, at))
    no_trunc_mean = np.mean(no_trunc_f1s) if no_trunc_f1s else 0
    best_f1 = no_trunc_mean

    # Try truncation at various percentiles of reference length
    for pct in [50, 75, 90, 100, 110, 125, 150, 200]:
        target_len = int(np.percentile(ref_lens, min(pct, 100)) * (pct/100 if pct > 100 else 1))
        trunc_f1s = []
        for i in sub_idx:
            ref = str(val_df.iloc[i]['output']).strip()
            rt = uni_toks(ref)
            try:
                ans = val_cands_all[i][0]['answer'] if val_cands_all[i] else ''
            except:
                ans = ''
            at = uni_toks(ans)
            if rt and at:
                truncated = at[:target_len]
                trunc_f1s.append(uni_r1(rt, truncated))
        trunc_mean = np.mean(trunc_f1s) if trunc_f1s else 0
        if trunc_mean > best_f1 + 0.001:  # need meaningful improvement
            best_f1 = trunc_mean
            best_trunc = target_len

    delta = best_f1 - no_trunc_mean
    marker = " ★" if delta > 0.005 else ""
    log(f"  {sub:<12} ref_len={np.median(ref_lens):>5.0f}  ans_len={np.median(ans_lens):>5.0f}  "
        f"R1_notrunc={no_trunc_mean:.4f}  R1_best={best_f1:.4f}  Δ={delta:+.4f}  "
        f"trunc={'none' if best_trunc<0 else best_trunc}{marker}")
print("\n")
# ================================================================
# PROPOSAL 3: SENTENCE-LEVEL EXTRACTIVE PRUNING
# ================================================================
log("="*60)
log("PROPOSAL 3: Sentence-Level Extractive Pruning")
log("="*60)
# Split sentences (handles multiple languages)
SENT_RE = re.compile(r'(?<=[.!?።\n])\s+')
def split_sentences(text):
    """Split text into sentences, handling Amharic (። ) and standard punctuation."""
    sents = SENT_RE.split(str(text).strip())
    return [s.strip() for s in sents if s.strip() and len(s.strip()) > 5]
def greedy_prune(answer, ref_toks, metric_fn=None):
    """Remove sentences that hurt ROUGE F1.
    Returns pruned answer if it improves F1, else original."""
    if metric_fn is None:
        metric_fn = uni_r1

    sents = split_sentences(answer)
    if len(sents) <= 1:
        return answer  # nothing to prune

    full_toks = uni_toks(answer)
    full_score = metric_fn(ref_toks, full_toks)

    best_score = full_score
    best_text = answer

    # Try removing each sentence, keep the removal that helps most
    improved = True
    current_sents = list(sents)

    while improved and len(current_sents) > 1:
        improved = False
        best_removal = -1
        for j in range(len(current_sents)):
            candidate = ' '.join(current_sents[:j] + current_sents[j+1:])
            ct = uni_toks(candidate)
            if not ct:
                continue
            score = metric_fn(ref_toks, ct)
            if score > best_score + 0.001:  # must be meaningfully better
                best_score = score
                best_removal = j
                best_text = candidate
                improved = True

        if best_removal >= 0:
            current_sents = current_sents[:best_removal] + current_sents[best_removal+1:]

    return best_text
# Evaluate on val
prune_results = defaultdict(lambda: {'before_r1':[], 'after_r1':[], 'before_rl':[], 'after_rl':[], 'changed': 0})
for i in tqdm(range(len(val_df)), desc="P3: Pruning"):
    sub = str(val_df.iloc[i]['subset'])
    ref = str(val_df.iloc[i]['output']).strip()
    rt = uni_toks(ref)
    if not rt:
        continue

    try:
        if not val_cands_all[i]:
            continue
        answer = val_cands_all[i][0]['answer']
    except:
        continue

    at = uni_toks(answer)
    if not at:
        continue

    before_r1 = uni_r1(rt, at)
    before_rl = uni_rl(rt, at)

    # Only prune answers with >1 sentence
    sents = split_sentences(answer)
    if len(sents) <= 1:
        prune_results[sub]['before_r1'].append(before_r1)
        prune_results[sub]['after_r1'].append(before_r1)
        prune_results[sub]['before_rl'].append(before_rl)
        prune_results[sub]['after_rl'].append(before_rl)
        continue

    pruned = greedy_prune(answer, rt, uni_r1)
    pt = uni_toks(pruned)
    after_r1 = uni_r1(rt, pt)
    after_rl = uni_rl(rt, pt)

    prune_results[sub]['before_r1'].append(before_r1)
    prune_results[sub]['after_r1'].append(after_r1)
    prune_results[sub]['before_rl'].append(before_rl)
    prune_results[sub]['after_rl'].append(after_rl)
    if pruned != answer:
        prune_results[sub]['changed'] += 1
log(f"\n{'Sub':<12} {'Before R1':>10} {'After R1':>10} {'Δ R1':>8} {'Changed':>8}")
log("-"*55)
tw_before, tw_after = 0, 0
for sub in sorted(SUBSET_TO_LANG.keys()):
    pr = prune_results[sub]
    b = np.mean(pr['before_r1']) if pr['before_r1'] else 0
    a = np.mean(pr['after_r1']) if pr['after_r1'] else 0
    w = TEST_MIX.get(sub, 0)
    tw_before += w * b
    tw_after += w * a
    n = len(pr['before_r1'])
    changed = pr['changed']
    marker = " ★" if a - b > 0.003 else ""
    log(f"  {sub:<12} {b:>10.4f} {a:>10.4f} {a-b:>+8.4f}  {changed:>5}/{n}{marker}")
log(f"\n  Test-weighted: Before={tw_before:.4f}  After={tw_after:.4f}  Delta={tw_after-tw_before:+.5f}")
tw_delta_disc = (tw_after - tw_before) * 0.37  # R1 weight
log(f"  Weighted score impact (R1 only): {tw_delta_disc:+.5f}")
log(f"  GATE: {'PASS ✅' if tw_delta_disc >= 0.001 else 'FAIL ❌'} (threshold: 0.001)")
print("\n")
# ================================================================
# SUMMARY
# ================================================================
log("="*60)
log("SUMMARY OF ALL PROPOSALS")
log("="*60)
log(f"P1 Cross-English:   Eng_Gha top-1 Δ R1 = {np.mean(cross_top1_r1)-bl_r1_mean:+.4f} → weighted est {(np.mean(cross_top1_r1)-bl_r1_mean)*0.375*0.188*0.37:+.5f}")
log(f"P2 Length Calib:     See per-language table above (look for ★ markers)")
log(f"P3 Sent Pruning:    Weighted Δ = {tw_after-tw_before:+.5f} → score impact {tw_delta_disc:+.5f}")
log(f"\nNext step: If any proposal shows PASS ✅, apply to test and build submission.")
log(f"If all FAIL ❌, lock V4+V2 as final selections.")


Using existing ROUGE functions
[06:03:08] ============================================================
[06:03:08] PROPOSAL 1: Cross-Subset English Retrieval for Eng_Gha
[06:03:08] ============================================================
[06:03:08] English pool: 21808 answers (Eng_Gha:5547, Eng_Uga:9312, Eng_Ken:2470, Eng_Eth:4479)
[06:03:08] Eng_Gha only: 5547 answers
[06:03:08] Expansion: 3.9x more candidates
[06:03:08] English embeddings: (21808, 1024)
[06:03:08] English FAISS index: 21808 vectors
[06:03:09] Eng_Gha val queries: 1104


P1: Cross-English: 100%|██████████| 1104/1104 [00:26<00:00, 41.54it/s]


[06:03:35] 
Eng_Gha Results (top-1 for baseline, oracle-top-5 for cross):
[06:03:35]   Baseline (same-subset): R1=0.3331  RL=0.2096
[06:03:35]   Cross-English (best-5): R1=0.4025  RL=0.2548
[06:03:35]   Delta:                  R1=+0.0694  RL=+0.0452
[06:03:35] 
  Best answer sources: {np.str_('Eng_Gha'): 1058, np.str_('Eng_Ken'): 7, np.str_('Eng_Uga'): 38, np.str_('Eng_Eth'): 1}
[06:03:35]   Answers from OTHER subsets: 46/1104
[06:03:35] 
  Raw R1 delta on Eng_Gha: +0.0694
[06:03:35]   Discounted (37.5%):     +0.0260
[06:03:35]   Test-weighted impact:   +0.00181
[06:03:35]   GATE: FAIL ❌ (threshold: 0.002)
[06:04:00] 
  Cross-English TOP-1 R1: 0.3355 (vs baseline 0.3331, delta +0.0024)


[06:04:00] ============================================================
[06:04:00] PROPOSAL 2: Per-Language Answer Length Calibration
[06:04:00] ============================================================
[06:04:01]   Aka_Gha      ref_len=   96  ans_len=   96  R1_notrunc=0.3305  R1_best=0.3319  Δ=+0.0

P3: Pruning: 100%|██████████| 6686/6686 [00:05<00:00, 1136.35it/s]

[06:04:15] 
Sub           Before R1   After R1     Δ R1  Changed
[06:04:15] -------------------------------------------------------
[06:04:15]   Aka_Gha          0.3305     0.3552  +0.0246    539/1114 ★
[06:04:15]   Amh_Eth          0.2076     0.2185  +0.0109    134/462 ★
[06:04:15]   Eng_Eth          0.6955     0.7180  +0.0225    161/564 ★
[06:04:15]   Eng_Gha          0.3331     0.3559  +0.0228    602/1104 ★
[06:04:15]   Eng_Ken          0.8991     0.9057  +0.0065     30/390 ★
[06:04:15]   Eng_Uga          0.8707     0.8765  +0.0058    134/1688 ★
[06:04:15]   Lug_Uga          0.8256     0.8322  +0.0066     96/846 ★
[06:04:15]   Swa_Ken          0.9484     0.9501  +0.0017     15/518
[06:04:15] 
  Test-weighted: Before=0.6509  After=0.6638  Delta=+0.01285
[06:04:15]   Weighted score impact (R1 only): +0.00475
[06:04:15]   GATE: PASS ✅ (threshold: 0.001)


[06:04:15] ============================================================
[06:04:15] SUMMARY OF ALL PROPOSALS
[06:04:15] =============

In [4]:
# ================================================================
# PROPOSAL 3 → TEST IMPLEMENTATION
# Sentence pruning using CONSENSUS proxy (no reference needed)
# ================================================================
# Run AFTER bootstrap. Uses cached val_cands_all, test candidates, etc.
import re, json, time, numpy as np, pandas as pd
from collections import defaultdict, Counter
from pathlib import Path
from tqdm import tqdm
log = lambda msg: print(f"[{time.strftime('%H:%M:%S')}] {msg}")
UNI_RE = re.compile(r'\w+', re.UNICODE)
SENT_RE = re.compile(r'(?<=[.!?።\n])\s+')
def uni_toks_local(text):
    return UNI_RE.findall(str(text).lower())
def uni_r1_local(ref_toks, hyp_toks):
    if not ref_toks or not hyp_toks: return 0.0
    rc = Counter(ref_toks); hc = Counter(hyp_toks)
    overlap = sum(min(rc[t], hc[t]) for t in rc)
    p = overlap / len(hyp_toks); r = overlap / len(ref_toks)
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0
# Use existing if available
try: uni_toks; uni_r1
except NameError: uni_toks = uni_toks_local; uni_r1 = uni_r1_local
def split_sents(text):
    sents = SENT_RE.split(str(text).strip())
    return [s.strip() for s in sents if s.strip() and len(s.strip()) > 5]
# ================================================================
# CONSENSUS-BASED PRUNING (reference-free)
# For each answer, remove sentences that have lowest overlap
# with OTHER top-K candidates' answers (the consensus)
# ================================================================
def build_pseudo_ref(candidates, exclude_idx=0):
    """Build a pseudo-reference from top-K candidates (excluding the selected one).
    Concatenate all other candidates to form a token bag."""
    tokens = []
    for j, c in enumerate(candidates):
        if j == exclude_idx:
            continue
        tokens.extend(uni_toks(c.get('answer', '')))
    return tokens
def consensus_prune(answer, candidates, min_sents=1):
    """Remove sentences from answer that have lowest overlap with other candidates.
    Reference-free: uses top-K consensus as proxy for reference."""
    sents = split_sents(answer)
    if len(sents) <= min_sents:
        return answer

    # Build pseudo-reference from other candidates
    pseudo_ref = build_pseudo_ref(candidates, exclude_idx=0)
    if not pseudo_ref:
        return answer

    # Also use query overlap as secondary signal
    full_toks = uni_toks(answer)
    full_score = uni_r1(pseudo_ref, full_toks)

    # Greedily remove sentences that improve consensus F1
    current_sents = list(sents)
    best_score = full_score
    best_text = answer

    improved = True
    while improved and len(current_sents) > min_sents:
        improved = False
        best_removal = -1
        for j in range(len(current_sents)):
            candidate_text = ' '.join(current_sents[:j] + current_sents[j+1:])
            ct = uni_toks(candidate_text)
            if not ct:
                continue
            score = uni_r1(pseudo_ref, ct)
            if score > best_score + 0.002:  # threshold to avoid noise
                best_score = score
                best_removal = j
                best_text = candidate_text
                improved = True
        if best_removal >= 0:
            current_sents = current_sents[:best_removal] + current_sents[best_removal+1:]

    return best_text
# ================================================================
# VALIDATE ON VAL: Consensus pruning vs Oracle pruning
# ================================================================
log("VALIDATING: Consensus-based pruning (reference-free) on val")
log("="*60)
TEST_MIX = {'Eng_Uga':0.284,'Aka_Gha':0.188,'Eng_Gha':0.188,'Lug_Uga':0.143,
            'Swa_Ken':0.087,'Eng_Ken':0.064,'Amh_Eth':0.023,'Eng_Eth':0.023}
cons_results = defaultdict(lambda: {'before_r1':[], 'after_r1':[], 'changed': 0})
for i in tqdm(range(len(val_df)), desc="Consensus prune val"):
    sub = str(val_df.iloc[i]['subset'])
    ref = str(val_df.iloc[i]['output']).strip()
    rt = uni_toks(ref)
    if not rt: continue

    try:
        cands = val_cands_all[i]
        if not cands: continue
        answer = cands[0]['answer']
    except:
        continue

    at = uni_toks(answer)
    if not at: continue

    before = uni_r1(rt, at)

    # Consensus prune (reference-free)
    pruned = consensus_prune(answer, cands)
    pt = uni_toks(pruned)
    after = uni_r1(rt, pt)

    cons_results[sub]['before_r1'].append(before)
    cons_results[sub]['after_r1'].append(after)
    if pruned != answer:
        cons_results[sub]['changed'] += 1
log(f"\n{'Sub':<12} {'Before':>8} {'After':>8} {'Δ R1':>8} {'Changed':>10}")
log("-"*55)
tw_before, tw_after = 0, 0
for sub in sorted(SUBSET_TO_LANG.keys()):
    cr = cons_results[sub]
    b = np.mean(cr['before_r1']) if cr['before_r1'] else 0
    a = np.mean(cr['after_r1']) if cr['after_r1'] else 0
    w = TEST_MIX.get(sub, 0)
    tw_before += w * b; tw_after += w * a
    n = len(cr['before_r1']); c = cr['changed']
    marker = " ★" if a - b > 0.003 else ""
    log(f"  {sub:<12} {b:>8.4f} {a:>8.4f} {a-b:>+8.4f} {c:>5}/{n}{marker}")
delta = tw_after - tw_before
score_impact = delta * 0.37
log(f"\n  Test-weighted R1: Before={tw_before:.4f}  After={tw_after:.4f}  Δ={delta:+.5f}")
log(f"  Score impact (R1 only): {score_impact:+.5f}")
# Apply Gha transfer discount
# Strong languages transfer ~1:1, Gha ~25-50%
# But pruning helps ALL languages, so blended discount is lighter
log(f"  After blended transfer discount (~70%): {score_impact*0.7:+.5f}")
log(f"  GATE: {'PASS ✅' if score_impact*0.7 >= 0.001 else 'FAIL ❌'}")
# ================================================================
# IF GATE PASSES: Apply to test and build submission
# ================================================================
if score_impact * 0.7 >= 0.001:
    log(f"\n{'='*60}")
    log("APPLYING CONSENSUS PRUNING TO TEST SET")
    log("="*60)

    # Load V4 as base
    import os
    v4_path = os.path.expanduser('~/Downloads/submission_v4_final.csv')
    v4 = pd.read_csv(v4_path)
    test_df_local = pd.read_csv('/home/mwaniasamuel/multilingual-health-qa/data/raw/Test.csv')

    pruned_answers = []
    changes_by_lang = defaultdict(int)
    total_by_lang = defaultdict(int)

    for i in tqdm(range(len(test_df_local)), desc="Pruning test"):
        rid = str(test_df_local.iloc[i]['ID'])
        sub = str(test_df_local.iloc[i]['subset'])
        total_by_lang[sub] += 1

        # Get current V4 answer
        v4_row = v4[v4['ID'] == rid]
        if len(v4_row) == 0:
            pruned_answers.append("No answer available.")
            continue

        answer = str(v4_row.iloc[0]['TargetR1F1']).strip()

        # Get candidates for this query (need test_cands or equivalent)
        try:
            cands = get_same_lang_candidates(
                str(test_df_local.iloc[i]['input']).strip(),
                test_emb[i], sub, k=5, exclude_exact=False
            )
        except:
            cands = [{'answer': answer}]

        if len(cands) < 2:
            pruned_answers.append(answer)
            continue

        # Consensus prune
        pruned = consensus_prune(answer, cands)
        if pruned != answer:
            changes_by_lang[sub] += 1
        pruned_answers.append(pruned)

    log(f"\nTest pruning changes by language:")
    for sub in sorted(total_by_lang.keys()):
        c = changes_by_lang.get(sub, 0)
        t = total_by_lang[sub]
        log(f"  {sub:<12} {c:>4}/{t:>4} changed ({100*c/t:.1f}%)")

    # Build submission
    sub_df = v4.copy()
    sub_df['TargetR1F1'] = pruned_answers
    sub_df['TargetRLF1'] = pruned_answers  # identical columns (safe)
    sub_df['TargetLLM'] = pruned_answers

    out_path = os.path.expanduser('~/Downloads/submission_v6_pruned.csv')
    sub_df.to_csv(out_path, index=False)
    log(f"\n✅ Saved: {out_path}")
    log(f"This is V4 + consensus sentence pruning. Identical columns.")
    log(f"Expected improvement: +{score_impact*0.7:.4f} on score (conservative)")
else:
    log("\n❌ Consensus pruning below threshold. Do NOT submit.")
    log("Lock V4+V2 as final selections.")


[06:16:12] VALIDATING: Consensus-based pruning (reference-free) on val
[06:16:12] ============================================================


Consensus prune val: 100%|██████████| 6686/6686 [00:19<00:00, 336.71it/s]

[06:16:32] 
Sub            Before    After     Δ R1    Changed
[06:16:32] -------------------------------------------------------
[06:16:32]   Aka_Gha        0.3305   0.3305  +0.0000     0/1114
[06:16:32]   Amh_Eth        0.2076   0.2076  +0.0000     1/462
[06:16:32]   Eng_Eth        0.6955   0.6955  +0.0000     0/564
[06:16:32]   Eng_Gha        0.3331   0.3331  +0.0000     0/1104
[06:16:32]   Eng_Ken        0.8991   0.8991  +0.0000     0/390
[06:16:32]   Eng_Uga        0.8707   0.8707  +0.0000     1/1688
[06:16:32]   Lug_Uga        0.8256   0.8256  +0.0000     0/846
[06:16:32]   Swa_Ken        0.9484   0.9484  +0.0000     0/518
[06:16:32] 
  Test-weighted R1: Before=0.6509  After=0.6509  Δ=+0.00000
[06:16:32]   Score impact (R1 only): +0.00000
[06:16:32]   After blended transfer discount (~70%): +0.00000
[06:16:32]   GATE: FAIL ❌
[06:16:32] 
❌ Consensus pruning below threshold. Do NOT submit.
[06:16:32] Lock V4+V2 as final selections.


In [5]:
# ================================================================
# LAST ATTEMPT: Query-based sentence pruning
# Remove sentences from answer that have ZERO overlap with the question
# ================================================================
import re, time, numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm
log = lambda msg: print(f"[{time.strftime('%H:%M:%S')}] {msg}")
UNI_RE = re.compile(r'\w+', re.UNICODE)
SENT_RE = re.compile(r'(?<=[.!?።\n])\s+')
STOPWORDS = {'the','a','an','is','are','was','were','be','been','being','have','has','had',
             'do','does','did','will','would','shall','should','may','might','can','could',
             'and','or','but','if','of','in','on','at','to','for','with','by','from','as',
             'this','that','these','those','it','its','they','them','their','we','our','you',
             'your','i','my','me','he','she','his','her','what','which','who','how','when',
             'where','why','not','no','yes','so','very','just','also','about','than','then',
             'into','over','after','before','between','under','through','during','against',
             'na','ya','wa','ni','kwa','la','za','au','je','si','no','ngo','mu','oku','ba',
             'ne','wo','no','a','ɛ','sɛ','ho','de','wɔ','bɛ','ɛno','ɛna','ane','na','nso'}
def content_toks(text):
    """Get content tokens (non-stopword) from text."""
    return [t for t in UNI_RE.findall(str(text).lower()) if t not in STOPWORDS and len(t) > 2]
def split_sents(text):
    sents = SENT_RE.split(str(text).strip())
    return [s.strip() for s in sents if s.strip() and len(s.strip()) > 5]
def uni_toks_l(text): return UNI_RE.findall(str(text).lower())
def uni_r1_l(ref_toks, hyp_toks):
    if not ref_toks or not hyp_toks: return 0.0
    rc = Counter(ref_toks); hc = Counter(hyp_toks)
    overlap = sum(min(rc[t], hc[t]) for t in rc)
    p, r = overlap/len(hyp_toks), overlap/len(ref_toks)
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0
try: uni_toks; uni_r1
except: uni_toks = uni_toks_l; uni_r1 = uni_r1_l
TEST_MIX = {'Eng_Uga':0.284,'Aka_Gha':0.188,'Eng_Gha':0.188,'Lug_Uga':0.143,
            'Swa_Ken':0.087,'Eng_Ken':0.064,'Amh_Eth':0.023,'Eng_Eth':0.023}
# ================================================================
# Method A: Remove sentences with zero content-word query overlap
# ================================================================
def query_prune_zero_overlap(answer, query):
    """Remove sentences that share zero content words with the query."""
    sents = split_sents(answer)
    if len(sents) <= 1:
        return answer

    q_content = set(content_toks(query))
    if not q_content:
        return answer

    kept = []
    for s in sents:
        s_content = set(content_toks(s))
        if s_content & q_content:  # at least one shared content word
            kept.append(s)

    if not kept:
        return answer  # safety: keep everything if all would be removed

    return ' '.join(kept)
# ================================================================
# Method B: Greedy removal using QUERY as pseudo-reference
# ================================================================
def query_prune_greedy(answer, query):
    """Greedily remove sentences that improve F1 overlap with query."""
    sents = split_sents(answer)
    if len(sents) <= 1:
        return answer

    qt = uni_toks(query)
    if not qt:
        return answer

    at = uni_toks(answer)
    full_score = uni_r1(qt, at)

    current_sents = list(sents)
    best_score = full_score
    best_text = answer

    improved = True
    while improved and len(current_sents) > 1:
        improved = False
        best_removal = -1
        for j in range(len(current_sents)):
            candidate = ' '.join(current_sents[:j] + current_sents[j+1:])
            ct = uni_toks(candidate)
            if not ct: continue
            score = uni_r1(qt, ct)
            if score > best_score + 0.005:
                best_score = score
                best_removal = j
                best_text = candidate
                improved = True
        if best_removal >= 0:
            current_sents = current_sents[:best_removal] + current_sents[best_removal+1:]

    return best_text
# ================================================================
# Method C: Keep only top-N sentences by query relevance
# ================================================================
def query_prune_topn(answer, query, keep_ratio=0.75):
    """Keep only the most query-relevant sentences."""
    sents = split_sents(answer)
    if len(sents) <= 2:
        return answer

    qt = set(content_toks(query))
    if not qt:
        return answer

    scored = []
    for s in sents:
        st = set(content_toks(s))
        overlap = len(st & qt)
        scored.append((overlap, s))

    # Keep at least keep_ratio of sentences, always keeping those with overlap > 0
    n_keep = max(1, int(len(sents) * keep_ratio))
    scored.sort(key=lambda x: -x[0])

    # But preserve original order
    kept_sents = set(s for _, s in scored[:n_keep])
    result = [s for s in sents if s in kept_sents]

    return ' '.join(result) if result else answer
# ================================================================
# EVALUATE ALL METHODS ON VAL
# ================================================================
log("="*60)
log("QUERY-BASED PRUNING — 3 Methods on Val")
log("="*60)
methods = {
    'A: zero-overlap': query_prune_zero_overlap,
    'B: greedy-query':  query_prune_greedy,
    'C: top-75%':       lambda a, q: query_prune_topn(a, q, 0.75),
}
for method_name, prune_fn in methods.items():
    results = defaultdict(lambda: {'before':[], 'after':[], 'changed':0})

    for i in tqdm(range(len(val_df)), desc=method_name):
        sub = str(val_df.iloc[i]['subset'])
        ref = str(val_df.iloc[i]['output']).strip()
        query = str(val_df.iloc[i]['input']).strip()
        rt = uni_toks(ref)
        if not rt: continue

        try:
            if not val_cands_all[i]: continue
            answer = val_cands_all[i][0]['answer']
        except: continue

        at = uni_toks(answer)
        if not at: continue

        before = uni_r1(rt, at)
        pruned = prune_fn(answer, query)
        pt = uni_toks(pruned)
        after = uni_r1(rt, pt) if pt else before

        results[sub]['before'].append(before)
        results[sub]['after'].append(after)
        if pruned != answer:
            results[sub]['changed'] += 1

    log(f"\n--- {method_name} ---")
    tw_b, tw_a = 0, 0
    for sub in sorted(SUBSET_TO_LANG.keys()):
        r = results[sub]
        b = np.mean(r['before']) if r['before'] else 0
        a = np.mean(r['after']) if r['after'] else 0
        w = TEST_MIX.get(sub, 0)
        tw_b += w*b; tw_a += w*a
        n = len(r['before']); c = r['changed']
        d = a - b
        marker = " ★" if d > 0.003 else (" ⚠️" if d < -0.003 else "")
        log(f"  {sub:<12} {b:.4f} → {a:.4f}  Δ={d:+.4f}  chg={c}/{n}{marker}")

    delta = tw_a - tw_b
    log(f"  Weighted: {tw_b:.4f} → {tw_a:.4f}  Δ={delta:+.5f}  score_impact={delta*0.37:+.5f}")
log(f"\n{'='*60}")
log("DONE. Compare methods. If any shows positive delta, build test submission.")
log("If all negative or zero, the predecessor was right. Lock V4+V2.")


[06:18:58] ============================================================
[06:18:58] QUERY-BASED PRUNING — 3 Methods on Val
[06:18:58] ============================================================


A: zero-overlap: 100%|██████████| 6686/6686 [00:07<00:00, 929.84it/s] 


[06:19:05] 
--- A: zero-overlap ---
[06:19:05]   Aka_Gha      0.3305 → 0.3186  Δ=-0.0119  chg=387/1114 ⚠️
[06:19:05]   Amh_Eth      0.2076 → 0.1974  Δ=-0.0103  chg=117/462 ⚠️
[06:19:05]   Eng_Eth      0.6955 → 0.5879  Δ=-0.1076  chg=384/564 ⚠️
[06:19:05]   Eng_Gha      0.3331 → 0.3149  Δ=-0.0182  chg=558/1104 ⚠️
[06:19:05]   Eng_Ken      0.8991 → 0.7994  Δ=-0.0998  chg=228/390 ⚠️
[06:19:05]   Eng_Uga      0.8707 → 0.7196  Δ=-0.1511  chg=1095/1688 ⚠️
[06:19:05]   Lug_Uga      0.8256 → 0.7641  Δ=-0.0615  chg=395/846 ⚠️
[06:19:05]   Swa_Ken      0.9484 → 0.8694  Δ=-0.0791  chg=225/518 ⚠️
[06:19:05]   Weighted: 0.6509 → 0.5776  Δ=-0.07335  score_impact=-0.02714


B: greedy-query: 100%|██████████| 6686/6686 [00:06<00:00, 1093.75it/s]


[06:19:11] 
--- B: greedy-query ---
[06:19:11]   Aka_Gha      0.3305 → 0.2538  Δ=-0.0768  chg=944/1114 ⚠️
[06:19:11]   Amh_Eth      0.2076 → 0.1968  Δ=-0.0108  chg=152/462 ⚠️
[06:19:11]   Eng_Eth      0.6955 → 0.5206  Δ=-0.1749  chg=395/564 ⚠️
[06:19:11]   Eng_Gha      0.3331 → 0.2561  Δ=-0.0770  chg=983/1104 ⚠️
[06:19:11]   Eng_Ken      0.8991 → 0.4910  Δ=-0.4081  chg=344/390 ⚠️
[06:19:11]   Eng_Uga      0.8707 → 0.4612  Δ=-0.4096  chg=1348/1688 ⚠️
[06:19:11]   Lug_Uga      0.8256 → 0.4236  Δ=-0.4020  chg=691/846 ⚠️
[06:19:11]   Swa_Ken      0.9484 → 0.4855  Δ=-0.4629  chg=462/518 ⚠️
[06:19:11]   Weighted: 0.6509 → 0.3776  Δ=-0.27338  score_impact=-0.10115


C: top-75%: 100%|██████████| 6686/6686 [00:02<00:00, 2468.41it/s]

[06:19:14] 
--- C: top-75% ---
[06:19:14]   Aka_Gha      0.3305 → 0.3199  Δ=-0.0107  chg=788/1114 ⚠️
[06:19:14]   Amh_Eth      0.2076 → 0.2036  Δ=-0.0040  chg=71/462 ⚠️
[06:19:14]   Eng_Eth      0.6955 → 0.6258  Δ=-0.0697  chg=244/564 ⚠️
[06:19:14]   Eng_Gha      0.3331 → 0.3244  Δ=-0.0087  chg=820/1104 ⚠️
[06:19:14]   Eng_Ken      0.8991 → 0.7984  Δ=-0.1008  chg=250/390 ⚠️
[06:19:14]   Eng_Uga      0.8707 → 0.7608  Δ=-0.1100  chg=1212/1688 ⚠️
[06:19:14]   Lug_Uga      0.8256 → 0.7335  Δ=-0.0921  chg=583/846 ⚠️
[06:19:14]   Swa_Ken      0.9484 → 0.8421  Δ=-0.1063  chg=337/518 ⚠️
[06:19:14]   Weighted: 0.6509 → 0.5855  Δ=-0.06544  score_impact=-0.02421
[06:19:14] 
[06:19:14] DONE. Compare methods. If any shows positive delta, build test submission.
[06:19:14] If all negative or zero, the predecessor was right. Lock V4+V2.
